# RetinaNet model for snowpole detection with LiDar and RGB datasets

### Import Packages

In [2]:
import os
import pandas as pd
from pathlib import Path
from PIL import Image

### Convert Yolo dataset format to RetinaNet format

In [6]:
# Create symlinks for RGB
!mkdir -p retinanet_data/rgb/images/train retinanet_data/rgb/labels/train
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/train/* retinanet_data/rgb/images/train/
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/labels/train/* retinanet_data/rgb/labels/train/

!mkdir -p retinanet_data/rgb/images/valid retinanet_data/rgb/labels/valid
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/valid/* retinanet_data/rgb/images/valid/
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/labels/valid/* retinanet_data/rgb/labels/valid/

!mkdir -p retinanet_data/rgb/images/test retinanet_data/rgb/labels/test
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/test/* retinanet_data/rgb/images/test/

# Create symlinks for LIDAR
!mkdir -p retinanet_data/lidar/images/train retinanet_data/lidar/labels/train
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/train/* retinanet_data/lidar/images/train/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/labels/train/* retinanet_data/lidar/labels/train/

!mkdir -p retinanet_data/lidar/images/valid retinanet_data/lidar/labels/valid
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/valid/* retinanet_data/lidar/images/valid/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/labels/valid/* retinanet_data/lidar/labels/valid/

!mkdir -p retinanet_data/lidar/images/test retinanet_data/lidar/labels/test
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/test/* retinanet_data/lidar/images/test/


In [8]:
def convert_yolo_to_retinanet_csv(image_dir, label_dir, output_csv, class_csv, class_map):
    image_dir = Path(image_dir)
    label_dir = Path(label_dir)

    annotations = []

    for label_file in sorted(label_dir.glob('*.txt')):
        img_file = image_dir / (label_file.stem + '.PNG')
        if not img_file.exists():
            img_file = image_dir / (label_file.stem + '.png')
        if not img_file.exists():
            print(f"Image for {label_file.stem} not found, skipping.")
            continue

        with Image.open(img_file) as img:
            img_w, img_h = img.size

        with open(label_file, 'r') as f:
            lines = f.readlines()

        if not lines:
            annotations.append([str(img_file), '', '', '', '', ''])
            continue

        for line in lines:
            class_id, x_center, y_center, width, height = map(float, line.strip().split())
            class_id = int(class_id)
            class_name = class_map.get(class_id, f'class_{class_id}')
            x1 = (x_center - width / 2) * img_w
            y1 = (y_center - height / 2) * img_h
            x2 = (x_center + width / 2) * img_w
            y2 = (y_center + height / 2) * img_h
            annotations.append([str(img_file), int(x1), int(y1), int(x2), int(y2), class_name])

    # Save annotations CSV
    df = pd.DataFrame(annotations, columns=['image_path', 'x1', 'y1', 'x2', 'y2', 'class_name'])
    df.to_csv(output_csv, index=False)
    print(f"Saved {len(df)} annotations to {output_csv}")

    # Save class mapping CSV
    with open(class_csv, 'w') as f:
        for i, name in sorted(class_map.items()):
            f.write(f"{name},{i}\n")
    print(f"Saved class mapping to {class_csv}")



In [13]:
# Example usage
class_map = {
    0: 'pole',
    # Add more classes here if needed
}

convert_yolo_to_retinanet_csv(
    image_dir='retinanet_data/rgb/images/train',
    label_dir='retinanet_data/rgb/labels/train',
    output_csv='annotations/rgb_train_annotations.csv',
    class_csv='classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='retinanet_data/rgb/images/valid',
    label_dir='retinanet_data/rgb/labels/valid',
    output_csv='annotations/rgb_valid_annotations.csv',
    class_csv='classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='retinanet_data/rgb/images/test',
    label_dir='retinanet_data/rgb/labels/test',
    output_csv='annotations/rgb_test_annotations.csv',
    class_csv='classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='retinanet_data/lidar/images/train',
    label_dir='retinanet_data/lidar/labels/train',
    output_csv='annotations/lidar_train_annotations.csv',
    class_csv='classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='retinanet_data/lidar/images/valid',
    label_dir='retinanet_data/lidar/labels/valid',
    output_csv='annotations/lidar_valid_annotations.csv',
    class_csv='classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='retinanet_data/lidar/images/test',
    label_dir='retinanet_data/lidar/labels/test',
    output_csv='annotations/lidar_test_annotations.csv',
    class_csv='classes.csv',
    class_map=class_map
)


FileNotFoundError: [Errno 2] No such file or directory: 'annotations/rgb_train_annotations.csv'